# FIA WEC 2021-2022 — Impacto das Condições Meteorológicas nos Tempos de Volta

**Dataset:** [FIA WEC Lap Data 2012-2022](https://www.kaggle.com/datasets/tristenterracciano/fia-wec-lap-data-20122022) + [Open-Meteo Historical Weather API](https://open-meteo.com)

**Research Question:** Como as condições meteorológicas (temperatura, precipitação, vento) influenciam os tempos de volta no Campeonato Mundial de Resistência da FIA (WEC)?

**Épocas analisadas:** 2017, 2021 e 2022  
**Corridas:** 18 (Silverstone, Spa, Le Mans, Nürburgring, Cidade do México, COTA, Fuji, Shanghai, Bahrain (6h e 8h em 2021), Portimão, Monza, Sebring)


## 1. Recolha de Dados Meteorológicos

Os dados meteorológicos são obtidos da API histórica do Open-Meteo para cada corrida, usando as coordenadas GPS de cada circuito. São recolhidos dados hora a hora de temperatura (°C), precipitação (mm/h) e velocidade do vento (kph).


In [ ]:
import os
import pandas as pd
import requests

def get_weather(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,precipitation,windspeed_10m"
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"Erro ao obter dados: {e}")
        return pd.DataFrame()

    df = pd.DataFrame({
        "time": data["hourly"]["time"],
        "temperature": data["hourly"]["temperature_2m"],
        "precipitation": data["hourly"]["precipitation"],
        "windspeed": data["hourly"]["windspeed_10m"]
    })
    return df


races = pd.DataFrame([
    {"season": 2017, "round": 1, "circuit": "SILVERSTONE",                              "start_date": "2017-04-16", "end_date": "2017-04-16", "lat": 52.0683, "lon": -1.0234},
    {"season": 2017, "round": 2, "circuit": "SPA FRANCORCHAMPS",                        "start_date": "2017-05-06", "end_date": "2017-05-06", "lat": 50.4372, "lon": 5.9714},
    {"season": 2017, "round": 3, "circuit": "LE MANS",                                  "start_date": "2017-06-18", "end_date": "2017-06-19", "lat": 47.9497, "lon": 0.2211},
    {"season": 2017, "round": 4, "circuit": "NURBURGRING",                              "start_date": "2017-07-16", "end_date": "2017-07-16", "lat": 50.3355, "lon": 6.9477},
    {"season": 2017, "round": 5, "circuit": "AUTODROMO HERMANOS RODRIGUEZ",             "start_date": "2017-09-03", "end_date": "2017-09-03", "lat": 19.4061, "lon": -99.0929},
    {"season": 2017, "round": 6, "circuit": "CIRCUIT OF THE AMERICAS",                  "start_date": "2017-09-16", "end_date": "2017-09-16", "lat": 30.1329, "lon": -97.6414},
    {"season": 2017, "round": 7, "circuit": "FUJI SPEEDWAY",                            "start_date": "2017-10-15", "end_date": "2017-10-15", "lat": 35.3721, "lon": 138.9271},
    {"season": 2017, "round": 8, "circuit": "SHANGHAI INTERNATIONAL CIRCUIT",           "start_date": "2017-11-05", "end_date": "2017-11-05", "lat": 31.3376, "lon": 121.2220},
    {"season": 2017, "round": 9, "circuit": "BAHRAIN INTERNATIONAL CIRCUIT",            "start_date": "2017-11-18", "end_date": "2017-11-18", "lat": 26.0325, "lon": 50.5106},

    {"season": 2021, "round": 1, "circuit": "SPA FRANCORCHAMPS",                        "start_date": "2021-05-01", "end_date": "2021-05-01", "lat": 50.4372, "lon": 5.9714},
    {"season": 2021, "round": 2, "circuit": "AUTODROMO DO ALGARVE",                     "start_date": "2021-06-13", "end_date": "2021-06-13", "lat": 37.2277, "lon": -8.6270},
    {"season": 2021, "round": 3, "circuit": "AUTODROMO NAZIONALE DI MONZA",             "start_date": "2021-07-18", "end_date": "2021-07-18", "lat": 45.6156, "lon": 9.2811},
    {"season": 2021, "round": 4, "circuit": "LE MANS",                                  "start_date": "2021-08-21", "end_date": "2021-08-22", "lat": 47.9497, "lon": 0.2211},
    {"season": 2021, "round": 5, "circuit": "BAHRAIN INTERNATIONAL CIRCUIT 6 HOURS",    "start_date": "2021-10-30", "end_date": "2021-10-30", "lat": 26.0325, "lon": 50.5106},
    {"season": 2021, "round": 6, "circuit": "BAHRAIN INTERNATIONAL CIRCUIT 8 HOURS",    "start_date": "2021-11-06", "end_date": "2021-11-06", "lat": 26.0325, "lon": 50.5106},

    {"season": 2022, "round": 1, "circuit": "SEBRING",                                  "start_date": "2022-03-17", "end_date": "2022-03-18", "lat": 27.4544, "lon": -81.3469},
    {"season": 2022, "round": 2, "circuit": "SPA FRANCORCHAMPS",                        "start_date": "2022-05-07", "end_date": "2022-05-07", "lat": 50.4372, "lon": 5.9714},
    {"season": 2022, "round": 3, "circuit": "LE MANS",                                  "start_date": "2022-06-11", "end_date": "2022-06-12", "lat": 47.9497, "lon": 0.2211},
])

os.makedirs("data", exist_ok=True)

weather_list = []
for _, race in races.iterrows():
    print(f"A obter dados: {race['season']} {race['circuit']}...")
    df_w = get_weather(race["lat"], race["lon"], race["start_date"], race["end_date"])
    df_w["season"] = race["season"]
    df_w["round"] = race["round"]
    df_w["circuit"] = race["circuit"]
    weather_list.append(df_w)

df_weather = pd.concat(weather_list, ignore_index=True)
df_weather["hour_numeric"] = pd.to_datetime(df_weather["time"]).dt.hour

df_weather.to_csv("data/weather.csv", index=False)
print(f"\n✅ Dados meteorológicos guardados: {len(df_weather)} linhas")
df_weather.head()


## 2. Limpeza e Integração dos Dados

O dataset WEC é carregado, limpo e integrado com os dados meteorológicos através de um merge por temporada, ronda, circuito e hora.

**Decisões de limpeza:**
- Strings vazias convertidas para NaN
- Voltas com pit stop removidas (distorcem os tempos)
- Outliers de `lap_time_s` removidos via IQR
- Colunas irrelevantes para a research question removidas
- Merge feito por hora para associar a temperatura correta a cada volta


In [ ]:
import pandas as pd
import numpy as np

df_wec = pd.read_csv("data/2012-2022_FIA_WEC_FULL_LAP_DATA.csv", low_memory=False)
df_weather = pd.read_csv("data/weather.csv")

print(f"WEC — Linhas: {len(df_wec)}, Colunas: {len(df_wec.columns)}")
print(f"Weather — Linhas: {len(df_weather)}, Colunas: {len(df_weather.columns)}")


In [ ]:
# Strings vazias → NaN
df_wec.replace("", pd.NA, inplace=True)

# Remover duplicados
df_wec.drop_duplicates(inplace=True)

# Colunas com significado próprio no vazio
df_wec["crossing_finish_line_in_pit"] = df_wec["crossing_finish_line_in_pit"].fillna("No Pit")
df_wec["pit_time"] = df_wec["pit_time"].fillna(0)
df_wec["group"] = df_wec["group"].fillna("N/A")
df_wec["flag_at_fl"] = df_wec["flag_at_fl"].fillna("UNKNOWN")

# top_speed → imputar com a média
df_wec["top_speed"] = df_wec["top_speed"].fillna(df_wec["top_speed"].mean())

# Remover linhas sem tempo de volta
df_wec.dropna(subset=["lap_time_s"], inplace=True)

# Remover voltas com pit stop
df_wec = df_wec[df_wec["crossing_finish_line_in_pit"] == "No Pit"]

# Remover outliers via IQR
Q1 = df_wec["lap_time_s"].quantile(0.25)
Q3 = df_wec["lap_time_s"].quantile(0.75)
IQR = Q3 - Q1
df_wec = df_wec[(df_wec["lap_time_s"] >= Q1 - 1.5 * IQR) & (df_wec["lap_time_s"] <= Q3 + 1.5 * IQR)]

# Remover classe com poucos dados
df_wec = df_wec[df_wec["class"] != "INNOVATIVE CAR"]

print(f"Linhas após limpeza: {len(df_wec)}")

# Valores em falta após limpeza
missing = df_wec.isnull().sum()
print("\nMissing values relevantes:")
print(missing[missing > 0])


In [ ]:
# Filtrar temporadas 2017, 2021 e 2022
df_wec = df_wec[df_wec["season"].isin(["2017", "2021", "2022", 2017, 2021, 2022])]

# Converter tipos para merge
df_wec["season"] = df_wec["season"].astype(int)
df_wec["round"] = df_wec["round"].astype(int)
df_weather["season"] = df_weather["season"].astype(int)
df_weather["round"] = df_weather["round"].astype(int)

# Extrair hora para merge
df_wec["hour_numeric"] = pd.to_datetime(df_wec["hour"], format="%H:%M:%S.%f").dt.hour

# Merge por season, round, circuit e hora
df_merged = pd.merge(df_wec, df_weather, on=["season", "round", "circuit", "hour_numeric"], how="left")

# Selecionar colunas relevantes
colunas_manter = [
    "season", "round", "circuit", "class", "manufacturer", "team", "vehicle", "driver_name",
    "lap_number", "lap_time_s", "top_speed", "kph", "hour",
    "temperature", "precipitation", "windspeed"
]
df_merged = df_merged[colunas_manter]

print(f"Linhas após merge: {len(df_merged)}")
print(f"Colunas após merge: {len(df_merged.columns)}")

df_merged.to_csv("data/wec_clean.csv", index=False)
print("\n✅ Dataset limpo guardado em: data/wec_clean.csv")
df_merged.head()


## 3. Análise Exploratória (EDA)

A análise é feita circuito a circuito para evitar a distorção causada pela mistura de circuitos com características muito diferentes. Para cada circuito são calculadas correlações entre as variáveis meteorológicas e os tempos de volta, e comparados os tempos em condições secas vs chuva.


In [ ]:
import pandas as pd

df = pd.read_csv("data/wec_clean.csv", low_memory=False)
df["chuva"] = df["precipitation"] > 0

print("--- Estatísticas descritivas ---")
print(df[["lap_time_s", "temperature", "precipitation", "windspeed"]].describe().round(2))


In [ ]:
cols = ["lap_time_s", "temperature", "precipitation", "windspeed"]

for circuito, grupo in df.groupby("circuit"):
    print(f"\n{'=' * 40}")
    print(f"CIRCUITO: {circuito}")
    print(f"{'=' * 40}")

    print(f"  Voltas totais: {len(grupo)}")
    print(f"  Temperatura média: {grupo['temperature'].mean():.1f}°C")
    print(f"  Voltas com chuva: {grupo['chuva'].sum()} ({grupo['chuva'].mean()*100:.1f}%)")

    print("\n  Correlações com lap_time_s:")
    print(grupo[cols].corr()["lap_time_s"].drop("lap_time_s").round(3).to_string())

    if grupo["chuva"].any():
        print("\n  Seco vs Chuva:")
        print(grupo.groupby("chuva")["lap_time_s"].agg(["mean", "count"]).round(2))


## 4. Visualizações

Cinco visualizações que respondem diretamente à research question, analisando o impacto de temperatura, precipitação e vento nos tempos de volta por circuito.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

df = pd.read_csv("data/wec_clean.csv", low_memory=False)
os.makedirs("plots", exist_ok=True)
sns.set_theme(style="whitegrid")

df["chuva"] = df["precipitation"] > 0
df["Condição"] = df["chuva"].map({True: "Chuva", False: "Seco"})


### Gráfico 1 — Heatmap de Correlações

In [ ]:
cols = ["lap_time_s", "temperature", "precipitation", "windspeed"]
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[cols].corr().round(2), annot=True, fmt=".2f",
            cmap="coolwarm", linewidths=0.5, ax=ax)
ax.set_title("Heatmap de Correlações — Clima vs Tempos de Volta", fontweight="bold")
plt.tight_layout()
plt.savefig("plots/01_heatmap_correlacoes.png")
plt.show()


### Gráfico 2 — Temperatura vs Tempo de Volta por Circuito

In [ ]:
circuitos = df["circuit"].unique()
n = len(circuitos)
fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(16, 10))
axes = axes.flatten()
for i, circuito in enumerate(circuitos):
    sub = df[df["circuit"] == circuito]
    axes[i].scatter(sub["temperature"], sub["lap_time_s"], alpha=0.2, s=8, color="steelblue")
    axes[i].set_title(circuito[:25], fontweight="bold", fontsize=9)
    axes[i].set_xlabel("Temperatura (°C)", fontsize=8)
    axes[i].set_ylabel("Lap Time (s)", fontsize=8)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Temperatura vs Tempo de Volta por Circuito", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/02_scatter_temperatura_por_circuito.png")
plt.show()


### Gráfico 3 — Velocidade do Vento vs Tempo de Volta por Circuito

In [ ]:
fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(16, 10))
axes = axes.flatten()
for i, circuito in enumerate(circuitos):
    sub = df[df["circuit"] == circuito]
    axes[i].scatter(sub["windspeed"], sub["lap_time_s"], alpha=0.2, s=8, color="darkorange")
    axes[i].set_title(circuito[:25], fontweight="bold", fontsize=9)
    axes[i].set_xlabel("Vento (kph)", fontsize=8)
    axes[i].set_ylabel("Lap Time (s)", fontsize=8)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Velocidade do Vento vs Tempo de Volta por Circuito", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/03_scatter_vento_por_circuito.png")
plt.show()


### Gráfico 4 — Tempos de Volta: Seco vs Chuva por Circuito

In [ ]:
circuitos_com_chuva = df[df["Condição"] == "Chuva"]["circuit"].unique()
df_chuva = df[df["circuit"].isin(circuitos_com_chuva)]
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df_chuva, x="circuit", y="lap_time_s", hue="Condição",
            palette=["steelblue", "salmon"], ax=ax)
ax.set_title("Tempos de Volta: Seco vs Chuva por Circuito", fontweight="bold")
ax.set_xlabel("Circuito")
ax.set_ylabel("Tempo de Volta (segundos)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("plots/04_boxplot_seco_chuva_por_circuito.png")
plt.show()


### Gráfico 5 — Temperatura Média por Circuito

In [ ]:
temp_media = df.groupby("circuit")["temperature"].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(temp_media.index, temp_media.values, color="tomato")
ax.set_title("Temperatura Média por Circuito", fontweight="bold")
ax.set_xlabel("Circuito")
ax.set_ylabel("Temperatura Média (°C)")
for bar, val in zip(bars, temp_media.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.3,
            f"{val:.1f}°C", ha="center", fontsize=9)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("plots/05_temperatura_por_circuito.png")
plt.show()

print("✅ 5 gráficos guardados na pasta plots/")


## 5. Conclusões

### Principais descobertas

**Temperatura (-0.02 correlação global)**
A temperatura não mostrou correlação significativa com os tempos de volta. A análise por circuito confirma que o layout do circuito é o principal fator que determina os tempos — Bahrain é quente e tem circuitos curtos, Spa é frio e tem um circuito longo, o que cria uma falsa correlação quando os dados são analisados globalmente.

**Precipitação (-0.07 correlação global)**
A chuva não mostrou impacto significativo nos tempos de volta. A correlação ligeiramente negativa é contra-intuitiva mas pode ser explicada pelo efeito Safety Car — quando chove há mais Safety Cars, e quando a corrida recomeça os carros têm pneus novos e andam mais rápido. Uma limitação importante é que o merge por hora não captura o estado cumulativo da pista após chuva.

**Vento (0.17 correlação global)**
O vento é o fator meteorológico com maior correlação com os tempos de volta. Mais vento tende a associar-se a voltas ligeiramente mais lentas, o que faz sentido aerodinamicamente. Este é o resultado mais consistente e robusto do estudo.

### Limitações
- Dataset limitado a 18 corridas em 3 temporadas
- Merge por hora não captura estados de pista (molhada após chuva)
- Diferenças entre classes de carros e estratégias de corrida introduzem variação não relacionada com o clima
- Seria necessário um dataset maior e mais diversificado para tirar conclusões definitivasSonnet 4.6